In [ ]:
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.0.2 imbalanced-learn==0.8.1


# !pip install scikit-learn==1.0.2 imbalanced-learn==0.8.1
!pip install -U scikit-learn imbalanced-learn

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib

# Configuration
DATA_DIR = "/kaggle/input/potato-disease-data/PlantVillage - Copy"
CLASSES = ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load and label images
print("Loading image paths and labels...")
image_paths = []
labels = []
for class_idx, class_name in enumerate(CLASSES):
    class_dir = os.path.join(DATA_DIR, class_name)
    for img_name in os.listdir(class_dir):
        image_paths.append(os.path.join(class_dir, img_name))
        labels.append(class_idx)

# Shuffle once before splitting
image_paths = np.array(image_paths)
labels = np.array(labels)

# 70% train, 30% temp
X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    image_paths, labels, test_size=0.30, stratify=labels, random_state=42
)

# From 30% temp: 25% val, 5% test
val_ratio = 25 / (25 + 5)  # = 0.8333
X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp, test_size=(1 - val_ratio), stratify=y_temp, random_state=42
)

print(f"\nDataset sizes:")
print(f"Training: {len(X_train_paths)} images")
print(f"Validation: {len(X_val_paths)} images")
print(f"Test: {len(X_test_paths)} images")

# Load MobileNetV2
print("\nLoading MobileNetV2 for feature extraction...")
feature_extractor = MobileNetV2(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(224, 224, 3)
)

def batch_extract_features(image_paths, batch_size=32):
    features = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting features"):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = []
        for img_path in batch_paths:
            img = image.load_img(img_path, target_size=IMG_SIZE)
            x = image.img_to_array(img)
            x = preprocess_input(x)
            batch_images.append(x)
        batch_images = np.array(batch_images)
        batch_features = feature_extractor.predict(batch_images, verbose=0)
        features.extend(batch_features.reshape(len(batch_images), -1))
    return np.array(features)

# Feature extraction
print("\nProcessing training set...")
X_train_features = batch_extract_features(X_train_paths, BATCH_SIZE)
print("\nProcessing validation set...")
X_val_features = batch_extract_features(X_val_paths, BATCH_SIZE)
print("\nProcessing test set...")
X_test_features = batch_extract_features(X_test_paths, BATCH_SIZE)

# Apply SMOTE
print("\nBalancing classes with SMOTE...")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_features, y_train)

print("\nClass distribution after SMOTE:")
print(f"early-blight: {np.sum(np.array(y_train_res) == 0)}")
print(f"late-blight: {np.sum(np.array(y_train_res) == 1)}")
print(f"healthy: {np.sum(np.array(y_train_res) == 2)}")


# Train Random Forest with accuracy tracking
print("\nTraining Random Forest...")
# Varying number of trees to observe training & validation accuracy
tree_counts = [50, 100, 150, 200, 250]
train_accuracies = []
val_accuracies = []

print("\nTraining Random Forest with varying number of trees...")

for n_trees in tree_counts:
    rf = RandomForestClassifier(
        n_estimators=n_trees,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_res, y_train_res)

    train_acc = rf.score(X_train_res, y_train_res)
    val_acc = rf.score(X_val_features, y_val)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"{n_trees} trees -> Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")


# Store accuracy metrics
train_accuracies = []
val_accuracies = []
tree_counts = []

# Train in increments for accuracy tracking
n_estimators_increment = 20
n_steps = 200 // n_estimators_increment

for i in tqdm(range(1, n_steps + 1), desc="Training Progress"):
    n_trees = i * n_estimators_increment
    rf.n_estimators = n_trees
    rf.fit(X_train_res, y_train_res)

    
    # Calculate accuracies
    train_pred = rf.predict(X_train_res)
    val_pred = rf.predict(X_val_features)
    
    train_acc = accuracy_score(y_train_res, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    tree_counts.append(n_trees)
    
    print(f"Trees: {n_trees:3d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# Final evaluation on test set
print("\nFinal Evaluation on Test Set:")
y_pred = rf.predict(X_test_features)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASSES))

# Confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()

# Plot accuracy progression
plt.figure(figsize=(10, 6))
plt.plot(tree_counts, train_accuracies, label='Training Accuracy', marker='o')
plt.plot(tree_counts, val_accuracies, label='Validation Accuracy', marker='o')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Random Forest Training Progress with SMOTE')
plt.legend()
plt.grid(True)
plt.savefig('accuracy_curve.png', bbox_inches='tight')
plt.show()

# Save model and features
joblib.dump(rf, 'random_forest_smote_model.pkl')
np.savez('extracted_features.npz', 
         X_train=X_train_features,
         X_val=X_val_features,
         X_test=X_test_features,
         y_train=y_train,
         y_val=y_val,
         y_test=y_test)

print("\nModel saved as 'random_forest_smote_model.pkl'")
print("Features saved as 'extracted_features.npz'")

import random
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing import image

# Map class indices to class names
class_names = ['Early_blight', 'Late_blight', 'Healthy']

# Select 12 random test images
random_indices = random.sample(range(len(X_test_paths)), 12)
selected_paths = [X_test_paths[i] for i in random_indices]
selected_labels = [y_test[i] for i in random_indices]

# Prepare figure
plt.figure(figsize=(18, 10))

# Predict and display
for i, (img_path, true_label) in enumerate(zip(selected_paths, selected_labels)):
    # Load and preprocess image
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_preprocessed = preprocess_input(img_array)
    features = feature_extractor.predict(np.expand_dims(img_preprocessed, axis=0), verbose=0)
    
    # Predict with Random Forest
    prediction = rf.predict(features)[0]
    proba = rf.predict_proba(features)[0]
    confidence = np.max(proba) * 100

    # Plot image with prediction info
    plt.subplot(3, 4, i + 1)
    plt.imshow(np.array(img))  # <-- FIXED LINE
    plt.axis('off')
    plt.title(
        f"Actual: {class_names[true_label]}\nPredicted: {class_names[prediction]}\nConfidence: {confidence:.1f}%",
        fontsize=10
    )

plt.tight_layout()
plt.show()
